# Обучение моделей  

## импорт библиотек и датасета

In [14]:
import os
import glob
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.model_selection import GridSearchCV, cross_validate, RandomizedSearchCV
from scipy.stats import randint
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier 
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

In [2]:
df = pd.read_csv('./dataset/data_cleaning.csv')

display(df.head())
display(df.shape)

,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,ag_stage,bmi,age_years,age_groups
0,2,168,62.0,110,80,1,1,0,0,1,0,-1,22.0,50,2
1,1,156,85.0,140,90,3,1,0,0,1,1,1,35.0,55,2
2,1,165,64.0,130,70,3,1,0,0,0,1,-1,24.0,51,2
3,2,169,82.0,150,100,1,1,0,0,1,1,-1,29.0,48,1
4,1,156,56.0,100,60,1,1,0,0,0,0,-1,23.0,47,1


(69527, 15)

In [3]:
X, y = df.drop('cardio', axis=1), df['cardio']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)

In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) 

display(X_train)
display(X_test)

,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,ag_stage,bmi,age_years,age_groups
46731,1,151,64.0,150,90,2,1,0,0,1,1,28.0,62,2
30055,2,170,58.0,120,80,1,1,1,1,1,-1,20.0,55,2
54614,1,164,70.0,150,100,1,1,0,0,1,-1,26.0,61,2
39798,1,157,63.0,110,80,1,1,0,0,1,-1,26.0,40,1
36448,1,157,58.8,120,90,1,1,0,0,1,-1,24.0,49,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37194,2,169,58.0,100,80,1,1,1,1,1,-1,20.0,49,1
6265,1,160,60.0,120,80,1,1,0,0,1,-1,23.0,49,1
54886,1,158,56.0,120,80,1,1,0,0,1,-1,22.0,51,2
860,2,160,67.0,120,80,1,1,0,0,1,-1,26.0,50,2


,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,ag_stage,bmi,age_years,age_groups
49686,1,165,65.0,120,80,3,3,0,0,0,-1,24.0,54,2
47580,2,172,62.0,120,80,1,1,1,1,1,-1,21.0,39,1
17346,1,166,100.0,150,90,3,1,0,0,1,1,36.0,58,2
30830,1,159,74.0,130,80,1,1,0,0,1,-1,29.0,62,2
69296,2,172,85.0,128,78,1,1,1,0,1,-1,29.0,49,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53216,2,173,77.0,130,80,3,1,0,1,0,-1,26.0,60,2
22396,1,168,70.0,110,70,1,1,0,0,1,-1,25.0,52,2
65480,2,168,62.0,100,70,1,1,0,0,1,-1,22.0,56,2
41067,1,160,72.0,110,80,1,1,0,0,1,-1,28.0,49,1


In [6]:
if not os.path.exists('./models/'):
    os.mkdir('./models/')

if not glob.glob('./models/desisiontree*.pkl'):
    extended_param_grid = {
        'criterion': ['gini', 'entropy'],
        'splitter': ['best', 'random'],
        'max_depth': [3, 5, 7, 10, 15, 20, 25, None],  
        'max_features': [None, 'sqrt', 'log2', 0.3, 0.5, 0.7, 0.9],
        'min_samples_split': [2, 5, 10, 15, 20],  
        'min_samples_leaf': [1, 2, 3, 5, 7, 10],  
        'random_state': [RANDOM_STATE]
    }

    DTC = DecisionTreeClassifier()

    grid_search = GridSearchCV(
        DTC, 
        extended_param_grid, 
        cv=5, 
        verbose=1,
        scoring='accuracy',
        refit=True,  
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)
    
    best_tree = grid_search.best_estimator_
    
    importances = best_tree.feature_importances_
    bad_features = []
    for feature_name, importance in zip(X_train.columns, importances):
        if importance < 0.01:
            bad_features.append(feature_name)

    print(f"Маловажные фичи: {bad_features}")
    print(f"Количество маловажных фич: {len(bad_features)}")

    important_features = [col for col in X_train.columns if col not in bad_features]
    X_train_important = X_train[important_features]
    X_test_important = X_test[important_features]
    
    print(f"Исходное количество признаков: {X_train.shape[1]}")
    print(f"После отбора важных: {X_train_important.shape[1]}")

    print("\nОбучение финальной модели на важных признаках...")
    
    final_model = DecisionTreeClassifier(**grid_search.best_params_)
    final_model.fit(X_train_important, y_train)

    print("\nОценка модели с помощью кросс-валидации...")
    
    
    
    scoring = {
        'accuracy': 'accuracy',
        'precision': 'precision',  
        'recall': 'recall',        
        'f1': 'f1'                 
    }

    cv_results = cross_validate(
        final_model, 
        X_train_important, 
        y_train, 
        cv=5, 
        scoring=scoring,
        return_train_score=True
    )

    print("\nРезультаты кросс-валидации (5 fold) на важных признаках:")
    print("Метрика       | Train     | Test")
    print("-" * 55)
    
    for metric in scoring.keys():
        train_scores = cv_results[f'train_{metric}']
        test_scores = cv_results[f'test_{metric}']
        
        print(f"{metric:12} | {train_scores.mean():.4f} | {test_scores.mean():.4f}")

    model_info = {
    'model': final_model,
    'important_features': important_features,
    'bad_features': bad_features,
    'best_params': grid_search.best_params_,
    'cv_results': cv_results,
    'feature_importances': dict(zip(important_features, final_model.feature_importances_))
    }

    cv_accuracy_mean = cv_results['test_accuracy'].mean()
    filename = f'./models/desisiontree_cvacc_{cv_accuracy_mean:.4f}_features_{len(important_features)}.pkl'
    joblib.dump(model_info, filename)

    print(f"\nМодель сохранена в файл: {filename}")
    print(f"Средняя accuracy по кросс-валидации: {cv_accuracy_mean:.4f}")

In [10]:
if not glob.glob('./models/randomforest*.pkl'):


    param_dist = {
        'n_estimators': randint(100, 500),  # Вместо 100-100
        'criterion': ['gini', 'entropy'],
        'max_depth': [3, 5, 7, 10, 15, None],
        'max_features': ['sqrt', 'log2', 0.3, 0.5],
        'min_samples_split': randint(2, 20),
        'min_samples_leaf': randint(1, 10),
        'random_state': [RANDOM_STATE]
    }

    RFC = RandomForestClassifier()

    random_search = RandomizedSearchCV(
        RFC, 
        param_dist, 
        n_iter=50,  
        cv=3,       
        verbose=1,
        scoring='accuracy',
        refit=True,
        n_jobs=-1,
        random_state=RANDOM_STATE
    )

    
    random_search.fit(X_train, y_train)
    
    best_forest = random_search.best_estimator_  
    
    importances = best_forest.feature_importances_
    bad_features = []
    for feature_name, importance in zip(X_train.columns, importances):
        if importance < 0.01:
            bad_features.append(feature_name)

    print(f"Маловажные фичи: {bad_features}")
    print(f"Количество маловажных фич: {len(bad_features)}")

    important_features = [col for col in X_train.columns if col not in bad_features]
    X_train_important = X_train[important_features]
    X_test_important = X_test[important_features]
    
    print(f"Исходное количество признаков: {X_train.shape[1]}")
    print(f"После отбора важных: {X_train_important.shape[1]}")

    print("\nОбучение финальной модели на важных признаках...")
    
    
    final_model = RandomForestClassifier(**random_search.best_params_)
    final_model.fit(X_train_important, y_train)

    print("\nОценка модели с помощью кросс-валидации...")
    
    scoring = {
        'accuracy': 'accuracy',
        'precision': 'precision',  
        'recall': 'recall',        
        'f1': 'f1'                 
    }

    cv_results = cross_validate(
        final_model, 
        X_train_important, 
        y_train, 
        cv=5, 
        scoring=scoring,
        return_train_score=True
    )

    print("\nРезультаты кросс-валидации (5 fold) на важных признаках:")
    print("Метрика       | Train     | Test")
    print("-" * 55)
    
    for metric in scoring.keys():
        train_scores = cv_results[f'train_{metric}']
        test_scores = cv_results[f'test_{metric}']
        
        print(f"{metric:12} | {train_scores.mean():.4f} | {test_scores.mean():.4f}")

    model_info = {
        'model': final_model,
        'important_features': important_features,
        'bad_features': bad_features,
        'best_params': random_search.best_params_,  
        'cv_results': cv_results,
        'feature_importances': dict(zip(important_features, final_model.feature_importances_))
    }

    cv_accuracy_mean = cv_results['test_accuracy'].mean()
    filename = f'./models/randomforest_cvacc_{cv_accuracy_mean:.4f}_features_{len(important_features)}.pkl'
    joblib.dump(model_info, filename)

    print(f"\nМодель сохранена в файл: {filename}")
    print(f"Средняя accuracy по кросс-валидации: {cv_accuracy_mean:.4f}")

Fitting 3 folds for each of 50 candidates, totalling 150 fits
Маловажные фичи: ['gender', 'smoke', 'alco', 'active']
Количество маловажных фич: 4
Исходное количество признаков: 14
После отбора важных: 10

Обучение финальной модели на важных признаках...

Оценка модели с помощью кросс-валидации...

Результаты кросс-валидации (5 fold) на важных признаках:
Метрика       | Train     | Test
-------------------------------------------------------
accuracy     | 0.7473 | 0.7348
precision    | 0.7722 | 0.7581
recall       | 0.7026 | 0.6909
f1           | 0.7358 | 0.7229

Модель сохранена в файл: ./models/randomforest_cvacc_0.7348_features_10.pkl
Средняя accuracy по кросс-валидации: 0.7348


In [15]:
if not glob.glob('./models/gradientboostingclassifier*.pkl'):


    param_dist = {
        'n_estimators': randint(100, 500),
        'loss': ['log_loss', 'exponential'],
        'learning_rate': [0.05, 0.1, 0.15, 0.2],
        'criterion': ['friedman_mse', 'squared_error'],
        'max_depth': [3, 5, 7, 10, 15, None],
        'subsample': [0.8, 0.9, 1.0],
        'min_samples_split': randint(2, 20),
        'min_samples_leaf': randint(1, 10),
        'random_state': [RANDOM_STATE]
    }

    GBC = GradientBoostingClassifier()

    random_search = RandomizedSearchCV(
        GBC, 
        param_dist, 
        n_iter=50,  
        cv=3,       
        verbose=1,
        scoring='accuracy',
        refit=True,
        n_jobs=-1,
        random_state=RANDOM_STATE
    )

    
    random_search.fit(X_train, y_train)
    
    best_forest = random_search.best_estimator_  
    
    importances = best_forest.feature_importances_
    bad_features = []
    for feature_name, importance in zip(X_train.columns, importances):
        if importance < 0.01:
            bad_features.append(feature_name)

    print(f"Маловажные фичи: {bad_features}")
    print(f"Количество маловажных фич: {len(bad_features)}")

    important_features = [col for col in X_train.columns if col not in bad_features]
    X_train_important = X_train[important_features]
    X_test_important = X_test[important_features]
    
    print(f"Исходное количество признаков: {X_train.shape[1]}")
    print(f"После отбора важных: {X_train_important.shape[1]}")

    print("\nОбучение финальной модели на важных признаках...")
    
    
    final_model = GradientBoostingClassifier(**random_search.best_params_)
    final_model.fit(X_train_important, y_train)

    print("\nОценка модели с помощью кросс-валидации...")
    
    scoring = {
        'accuracy': 'accuracy',
        'precision': 'precision',  
        'recall': 'recall',        
        'f1': 'f1'                 
    }

    cv_results = cross_validate(
        final_model, 
        X_train_important, 
        y_train, 
        cv=5, 
        scoring=scoring,
        return_train_score=True
    )

    print("\nРезультаты кросс-валидации (5 fold) на важных признаках:")
    print("Метрика       | Train     | Test")
    print("-" * 55)
    
    for metric in scoring.keys():
        train_scores = cv_results[f'train_{metric}']
        test_scores = cv_results[f'test_{metric}']
        
        print(f"{metric:12} | {train_scores.mean():.4f} | {test_scores.mean():.4f}")

    model_info = {
        'model': final_model,
        'important_features': important_features,
        'bad_features': bad_features,
        'best_params': random_search.best_params_,  
        'cv_results': cv_results,
        'feature_importances': dict(zip(important_features, final_model.feature_importances_))
    }

    cv_accuracy_mean = cv_results['test_accuracy'].mean()
    filename = f'./models/gradientboostingclassifier_cvacc_{cv_accuracy_mean:.4f}_features_{len(important_features)}.pkl'
    joblib.dump(model_info, filename)

    print(f"\nМодель сохранена в файл: {filename}")
    print(f"Средняя accuracy по кросс-валидации: {cv_accuracy_mean:.4f}")

Fitting 3 folds for each of 50 candidates, totalling 150 fits
Маловажные фичи: ['gender', 'height', 'gluc', 'smoke', 'alco', 'active', 'ag_stage', 'age_groups']
Количество маловажных фич: 8
Исходное количество признаков: 14
После отбора важных: 6

Обучение финальной модели на важных признаках...

Оценка модели с помощью кросс-валидации...

Результаты кросс-валидации (5 fold) на важных признаках:
Метрика       | Train     | Test
-------------------------------------------------------
accuracy     | 0.7394 | 0.7339
precision    | 0.7542 | 0.7487
recall       | 0.7112 | 0.7053
f1           | 0.7321 | 0.7263

Модель сохранена в файл: ./models/gradientboostingclassifier_cvacc_0.7339_features_6.pkl
Средняя accuracy по кросс-валидации: 0.7339
